<a href="https://colab.research.google.com/github/codebysumit/cryptography-algorithms/blob/master/notebooks/vigenere_cipher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vigenere Cipher Technique

## History
The Vigenere Cipher is a polyalphabetic substitution cipher. It was first described by Giovan Battista Bellaso in 1553, but it got its name from Blaise de Vigenere, who was wrongly credited with it in the 19th century. For a long time people called it "le chiffre indechiffrable", which means "the indecipherable cipher", because it was very hard to break using simple frequency analysis. It stayed unbroken until Charles Babbage and later Friedrich Kasiski found ways to attack it in the 1800s.

## What is Vigenere Cipher Technique?
The Vigenere Cipher is different from the Caesar Cipher because it does not use one single shift value for the whole message. Instead, it uses a **keyword**. Each letter of the keyword gives a different shift value, and the keyword is repeated again and again until it covers the full length of the plaintext.

Because the shift value keeps changing letter by letter, the same plaintext character can encrypt into many different ciphertext characters depending on its position. This makes it a **polyalphabetic** cipher, and it is much stronger against simple frequency analysis than a monoalphabetic cipher like Caesar or Affine.

In this implementation, we use the printable ASCII range from space (` `) to tilde (`~`), which spans from ASCII value 32 to 126 ($N=95$). This means the cipher works on letters, digits, punctuation, and symbols, not just A-Z.

## Cryptography Algorithm

### Constants
*   $S = 32$ (Start of printable ASCII range)
*   $E = 126$ (End of printable ASCII range)
*   $N = E - S + 1 = 95$ (Total number of printable characters)
*   $K$ = Keyword string, repeated cyclically to match the length of the plaintext
*   $x$ = Numeric value of the plaintext character ($0 \le x < N$)
*   $y$ = Numeric value of the ciphertext character ($0 \le y < N$)
*   $k_i$ = Numeric value of the keyword character used at position $i$ ($0 \le k_i < N$)

### 1. Key Stream Generation
Since the keyword is usually shorter than the plaintext, it is repeated cyclically so every plaintext character gets a matching key character. For a plaintext of length $L$ and a keyword of length $M$, the key character used at position $i$ is:

$$k_i = K[\, i \bmod M \,]$$

### 2. Encryption
For each character in the plaintext, the character index $x$ at position $i$ is transformed into a ciphertext index $y$ using the formula:

$$E(x_i) = (x_i + k_i) \pmod N$$

To get the final ASCII value: $C = E(x_i) + S$

### 3. Decryption
To reverse the process, the decryption function subtracts the same key value that was used for encryption at that position:

$$D(y_i) = (y_i - k_i) \pmod N$$

To get the final ASCII value: $P = D(y_i) + S$

### Key Requirements
*   The **keyword** can be any non empty string made of printable characters.
*   Unlike the Affine Cipher, there is no coprime condition here, because the operation is addition/subtraction modulo $N$, not multiplication. Every keyword works.
*   A longer and more random keyword makes the cipher harder to break using Kasiski examination or frequency analysis.

### 1. Import Dependencies

In [ ]:
import random
import string

### 2. Helper Utilities

In [ ]:
def build_key_stream(text_length: int, key: str) -> str:
    # repeat the keyword cyclically until it matches the plaintext length
    key_stream = ""
    for i in range(text_length):
        key_stream += key[i % len(key)]
    return key_stream

### 3. Generate Random Keyword

In [ ]:
def generate_random_key(length: int = 6) -> str:
    # build a random keyword using letters, digits and a few symbols
    charset = string.ascii_letters + string.digits
    return "".join(random.choice(charset) for _ in range(length))

### 4. Encryption

In [ ]:
def encrypt(text: str, key: str) -> str:
    START_ASCII = 32
    END_ASCII = 126
    TOTAL_CHAR = END_ASCII - START_ASCII + 1  # 95 characters

    if not key:
        raise ValueError("'key' must be a non empty string.")

    key_stream = build_key_stream(len(text), key)
    encrypted_text = ""

    for i, ch in enumerate(text):
        code = ord(ch)
        if START_ASCII <= code <= END_ASCII:
            shift = ord(key_stream[i]) - START_ASCII
            cipher_code = ((code - START_ASCII) + shift) % TOTAL_CHAR
            encrypted_text += chr(cipher_code + START_ASCII)
        else:
            encrypted_text += ch

    return encrypted_text

### 5. Decryption

In [ ]:
def decrypt(text: str, key: str) -> str:
    START_ASCII = 32
    END_ASCII = 126
    TOTAL_CHAR = END_ASCII - START_ASCII + 1  # 95 characters

    if not key:
        raise ValueError("'key' must be a non empty string.")

    key_stream = build_key_stream(len(text), key)
    decrypted_text = ""

    for i, ch in enumerate(text):
        code = ord(ch)
        if START_ASCII <= code <= END_ASCII:
            shift = ord(key_stream[i]) - START_ASCII
            plain_code = ((code - START_ASCII) - shift) % TOTAL_CHAR
            decrypted_text += chr(plain_code + START_ASCII)
        else:
            decrypted_text += ch

    return decrypted_text

### 6. Example usage

In [30]:
key = generate_random_key(10)
print(f"Generated Random Key: {key}")

Generated Random Key: BvFPf4B3Gs


In [31]:
plaintext = """TOP secret Massage! Agent 101, visit Area 51 (37d14'0\"N 115d48'30\"W)."""
print(f"Original Plain Text: {plaintext}")

cipher_text = encrypt(plaintext, key)
print("Encrypted:", cipher_text)

decrypted_text = decrypt(cipher_text, key)
print("Decrypted:", decrypted_text)

match = plaintext == decrypted_text
print(f"Verification Match:{match}")

Original Plain Text: TOP secret Massage! Agent 101, visit Area 51 (37d14'0"N 115d48'30"W).
Encrypted: vFvPZy&&-hBD(DZu*xHsc^,?[4SCX Bm0DP)BT:Y$v[af<UJ,%V}VR54SD\XV/Mcv6y<U
Decrypted: TOP secret Massage! Agent 101, visit Area 51 (37d14'0"N 115d48'30"W).
Verification Match:True
